# diagonal-via-strides — worked example 1: Extract the k-th sub-diagonal via as_strided

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `diagonal-via-strides`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

For a row-major `(N, N)` tensor the row stride is `N` and the column stride is `1`, so a diagonal step (one row down, one column right) has stride `N + 1`. To read the k-th **sub**-diagonal `[m[k,0], m[k+1,1], ...]` the walk still uses stride `N + 1`, but it **starts** at row `k`, column `0`. That start sits at linear storage offset `k * N`, and the diagonal has length `N - k`.

## Worked solution

**Step 1 — pin down what a sub-diagonal is.** The k-th sub-diagonal lies *below* the main diagonal: `m[k,0], m[k+1,1], ..., m[N-1, N-1-k]`. Each element advances one row down and one column right from the previous, so it is a true diagonal walk — only the starting corner differs from the main diagonal.

**Step 2 — reuse the diagonal stride.** A single diagonal step adds `N` (down a row) plus `1` (right a column) to the linear index, so `stride = (N + 1,)`. This is identical to the main-diagonal and super-diagonal cases; the stride never changes for a 45-degree walk in a row-major matrix.

**Step 3 — compute the starting offset.** The first element is `m[k, 0]`. In row-major storage `m[k, 0]` lives at linear index `k * N + 0 = k * N`. So `storage_offset = k * N`. (Contrast the super-diagonal, which starts at `m[0, k]` → offset `k`.)

**Step 4 — compute the length.** Starting at row `k`, we can take steps until we run off the bottom or right edge. Rows `k..N-1` give `N - k` elements, so `size = (N - k,)`.

**Step 5 — assemble and verify.** `m.as_strided(size=(N - k,), stride=(N + 1,), storage_offset=k * N)` returns a no-copy view. We confirm it equals `torch.diagonal(m, offset=-k)`, which is PyTorch's built-in for sub-diagonals (negative offset).

In [ ]:
def kth_sub_diagonal(m: Tensor, k: int) -> Tensor:
    N = m.shape[0]
    return m.as_strided(size=(N - k,), stride=(N + 1,), storage_offset=k * N)


t.manual_seed(0)
N = 5
m = t.arange(N * N).reshape(N, N)
k = 2
d = kth_sub_diagonal(m, k)
ref = t.diagonal(m, offset=-k)
print("sub-diagonal:", d.tolist())
print("matches torch.diagonal:", bool(t.equal(d, ref)))
print("is a view (shares storage):", d.data_ptr() == m.data_ptr() + k * N * m.element_size())